In [1]:
from enum import Enum


class StrEnum(str, Enum):
    def __str__(self):
        return self.value


class Exchange(StrEnum):
    Polymarket = "POLYMARKET"
    Binance = "BINANCE"
    Bybit = "BYBIT"


class DatabaseTableNames(str, Enum):
    Binance = "binance_trades"
    Polymarket = "polymarket_trades"


class Timeframe(str, Enum):
    # Seconds
    S1 = "1s"
    S5 = "5s"
    S15 = "15s"
    S30 = "30s"

    # Minutes
    M1 = "1m"
    M3 = "3m"
    M5 = "5m"
    M15 = "15m"
    M30 = "30m"

    # Hours
    H1 = "1h"
    H2 = "2h"
    H4 = "4h"
    H6 = "6h"
    H12 = "12h"

    # Days
    D1 = "1d"
    D3 = "3d"
    D7 = "1w"


    def to_seconds(self) -> int:
        unit = self.value[-1]
        num = int(self.value[:-1])
        if unit == "s":
            return num
        elif unit == "m":
            return num * 60
        elif unit == "h":
            return num * 3600
        elif unit == "d":
            return num * 86400
        elif unit == "w":
            return num * 604800
        else:
            raise ValueError(f"Unknown timeframe: {self.value}")

class BinanceNewsCategory:
    LISTING = 48
    DELISTING = 161

class NewsCategory:
    LISTING = "listing"
    DELISTING = "delisting"


import os
from pydantic import BaseModel, ConfigDict


class OHLCVTransactionModel(BaseModel):
    model_config = ConfigDict(
        strict=True,
        # frozen=True,
    )

    timestamp: int
    open: float
    high: float
    low: float
    close: float
    volume: float
    datetime: str
    ticker: str



from pydantic import BaseModel, Field
import os


class DatabaseConfig(BaseModel):
    host: str = Field(default_factory=lambda: os.getenv("DB_HOST",  "localhost"))
    port: int = Field(default_factory=lambda: int(os.getenv("DB_PORT", 9000)))
    username: str = Field(default_factory=lambda: os.getenv("DB_USERNAME", "admin"))
    password: str = Field(default_factory=lambda: os.getenv("DB_PASSWORD", "quest"))
    auto_flush_rows: int = Field(default_factory=lambda: int(os.getenv("DB_AUTO_FLUSH_ROWS", 100)))
    auto_flush_interval: int = Field(default_factory=lambda: int(os.getenv("DB_AUTO_FLUSH_INTERVAL", 1000)))


    def get_connection_string(self) -> str:
        return (
            f"http::addr={self.host}:{self.port};"
            f"username={self.username};password={self.password};"
            f"auto_flush_rows={self.auto_flush_rows};"
            f"auto_flush_interval={self.auto_flush_interval};"
        )

In [2]:
db_conf = DatabaseConfig(port=9000).get_connection_string()
table_name = 'news'

In [3]:
import json

with open("../../../../sample_data/parsed_news_binance.json", "r") as f:
    parsed_binance_news = json.load(f)


import json

with open("../../../../sample_data/parsed_news_bybit.json", "r") as f:
    parsed_bybit_news = json.load(f)

In [5]:
from datetime import datetime
from questdb.ingress import Sender


def insert_news_to_db(
    db_config: str,
    table_name: str,
    rows: list[dict],
    exchange: Exchange,
    logger,
) -> None:

    with Sender.from_conf(db_config) as sender:

        for row in rows:

            sender.row(
                table_name=table_name,

                # Must match TIMESTAMP(start_timestamp)
                at=datetime.fromtimestamp(row["releaseDate"] / 1000),

                # SYMBOL columns
                symbols={
                    "exchange": exchange.value,
                    "type_event": row["type_event"],
                },

                # Regular columns
                columns={
                    "id": str(row["id"]),
                    "announcement": row["title"],
                },
            )

        sender.flush()

    logger.info(f"Inserted {len(rows)} news rows.")

In [41]:
import logging
insert_news_to_db(
    db_config=db_conf,
    table_name="dev_news",
    rows = parsed_bybit_news,
    exchange=Exchange.Bybit,
    logger=logging.getLogger("test")
)

In [6]:
import logging
insert_news_to_db(
    db_config=db_conf,
    table_name="dev_news",
    rows = parsed_binance_news,
    exchange=Exchange.Binance,
    logger=logging.getLogger("test")
)